# 第 23 章 · 从 TRPO 到 PPO：一批数据多用几次，但每次别改太多

本章回答：策略梯度每批数据只能走一步，太浪费；想多走几步，"数据是旧策略采的"怎么办？为什么要限制每次改动的幅度？PPO 的 clip 到底在剪什么？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/23-ppo.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch


print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 重要性采样例子 | `importance_weighted_mean([1, 0], [0.8, 0.2], [0.5, 0.5])` | 0.8 |
| 表格五行 | `ppo_term(ratio, adv)`、`grad_wrt_ratio(ratio, adv)` | +2.20/+2、+2.40/0、−0.90/−1、−0.80/0、+1.00/+2 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `ppo_term`**（来自 `ch23_ppo.py`）：下面要用到的小工具。

In [ ]:
def ppo_term(ratio, adv, eps=0.2):
    r = torch.as_tensor(ratio, dtype=torch.float32)
    return torch.minimum(r * adv, torch.clamp(r, 1 - eps, 1 + eps) * adv)

**函数 `grad_wrt_ratio`**（来自 `ch23_ppo.py`）：下面要用到的小工具。

In [ ]:
def grad_wrt_ratio(ratio, adv, eps=0.2):
    r = torch.tensor(float(ratio), requires_grad=True)
    ppo_term(r, adv, eps).backward()
    return r.grad.item()

**函数 `importance_weighted_mean`**（来自 `ch23_ppo.py`）：重要性采样：用旧策略的样本估新策略的平均，每个样本乘 p_新 / p_旧。

In [ ]:
def importance_weighted_mean(rewards, p_new, p_old):
    """重要性采样：用旧策略的样本估新策略的平均，每个样本乘 p_新 / p_旧。"""
    return sum(r * pn / po for r, pn, po in zip(rewards, p_new, p_old)) / len(rewards)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("重要性采样例子 =", importance_weighted_mean([1, 0], [0.8, 0.2], [0.5, 0.5]))

In [ ]:
for ratio, adv in [(1.1, 2.0), (1.5, 2.0), (0.9, -1.0), (0.5, -1.0), (0.5, 2.0)]:
        print(f"r = {ratio}，A = {adv:+}：L = {ppo_term(ratio, adv).item():+.2f}，dL/dr = {grad_wrt_ratio(ratio, adv):+.1f}")

## 3. 代码里才看得到的

1. `torch.clamp` 在被剪住的区域梯度是 0，这正是"改过头就停"在代码里的实现方式，不需要额外写判断；
2. 实际代码里比率用 `exp(new_logprob − old_logprob)` 算（log 相减再取 e 次方），比直接相除数值更稳；`old_logprob` 必须在采样时存下来并 `detach()`，否则梯度会穿到旧策略里去。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 换 ε：比率 1.3、A = +1 时，`ppo_term(1.3, 1.0, eps)` 在 ε = 0.1 / 0.2 / 0.4 时是 1.1、1.2、1.3，梯度是 0、0、1：ε 越大，允许走得越远。
2. 比率 2.0、A = −1：三种 ε 下目标都是 −2.0，梯度都是 −1，一直没被剪。新策略把坏动作的概率加大了，剪裁不会挡住往回改的梯度。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 一批数据只走一步太浪费，想多走几步要先解决数据过期；
> 重要性采样：乘 新/旧，旧数据估新平均；比率叫 r；
> 比率别太极端、策略别跑太远：信任区域，KL 不超过 δ；
> PPO 剪比率：好的最多涨到 1.2 倍，坏的最多压到 0.8 倍；改对过头停，改错照拉。